# PDBe — Protein Data Bank in Europe

**PDBe** (Protein Data Bank in Europe) is the European resource for the collection, organisation and dissemination of data about biological macromolecular structures. PDBe is a founding member of the Worldwide PDB (wwPDB) partnership, responsible for annotating and distributing a quarter of all PDB entries.

| Property | Details |
|---|---|
| **Full name** | Protein Data Bank in Europe |
| **URL** | https://www.ebi.ac.uk/pdbe/ |
| **Number of structures** | ~220,000+ (and growing) |
| **Data types** | X-ray crystallography, NMR spectroscopy, cryo-electron microscopy (cryo-EM), neutron diffraction, electron crystallography |
| **Primary use cases** | Structural biology research, drug discovery, protein function annotation, evolutionary analysis, computational modelling benchmarking |

PDBe provides a comprehensive REST API and graph API for programmatic access, as well as PDBE-KB (Knowledge Base) aggregating functional annotations from UniProt, Pfam, CATH, and many other resources onto every residue of every structure.

In [ ]:
import json
import time
from pathlib import Path

import requests
import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to PDBe REST API and confirm access
    * [x] Fetch current PDB entry ID list from RCSB holdings API
    * [x] Retrieve entry summaries for a sample of structures via PDBe API
    * [x] Parse into a Polars DataFrame with key fields (pdb_id, title, experimental_method, resolution, organism)
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise DataFrame dimensions and null rates
    * [ ] Parse deposition/release dates as date types
    * [ ] Filter by experimental method (X-ray, NMR, EM)
    * [ ] Clean resolution values (handle NMR entries without resolution)
* [ ] **Analysis**
    * [ ] Count structures by experimental method
    * [ ] Track deposition rate over years
    * [ ] Identify most common source organisms
* [ ] **Visualization**
    * [ ] Bar chart of experimental methods
    * [ ] Line plot of annual depositions
    * [ ] Resolution distribution histogram for X-ray structures
* [ ] **Statistical analysis**
    * [ ] Compare resolution distributions across decades
    * [ ] Test for deposition rate trends

## 1. Ingest Data

### 1.1 Connect to PDBe API

In [ ]:
PDBE_BASE = "https://www.ebi.ac.uk/pdbe/api"

def pdbe_get(endpoint: str, timeout: int = 30) -> dict:
    """
    Send a GET request to the PDBe REST API.

    Parameters
    ----------
    endpoint : str
        Path after the base URL (e.g. "pdb/entry/summary/1cbs").
    timeout : int
        Request timeout in seconds.

    Returns
    -------
    dict
        Parsed JSON response body.
    """
    url = f"{PDBE_BASE}/{endpoint}"
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

# Connectivity check: fetch the summary for 1CBS (retinoic acid binding protein)
# This is a small, well-characterised X-ray structure from 1994 — a reliable test entry.
test_entry = "1cbs"
summary = pdbe_get(f"pdb/entry/summary/{test_entry}")

print(f"Connected to PDBe API successfully.")
print(f"Test entry: {test_entry.upper()}")

entry_data = summary[test_entry][0]
print(f"  Title      : {entry_data['title']}")
print(f"  Method     : {entry_data['experimental_method']}")
print(f"  Organism   : {entry_data['related_structures']}")
print(f"  Released   : {entry_data.get('release_date', 'N/A')}")
print(f"  Deposited  : {entry_data.get('deposition_date', 'N/A')}")

### 1.2 Fetch Full PDB Entry ID List

Download the complete list of current PDB entry IDs from the RCSB holdings API. This returns a JSON array of all ~220,000+ accession codes. Results are cached in `data/pdb_entry_ids.json` to avoid repeated downloads.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ENTRY_IDS_URL = "https://data.rcsb.org/rest/v1/holdings/current/entry_ids"
ENTRY_IDS_PATH = DATA_DIR / "pdb_entry_ids.json"

if ENTRY_IDS_PATH.exists():
    print(f"Loading cached entry IDs from {ENTRY_IDS_PATH}")
    with open(ENTRY_IDS_PATH) as f:
        entry_ids = json.load(f)
else:
    print(f"Downloading PDB entry ID list from RCSB ...")
    resp = requests.get(ENTRY_IDS_URL, timeout=60)
    resp.raise_for_status()
    entry_ids = resp.json()
    with open(ENTRY_IDS_PATH, "w") as f:
        json.dump(entry_ids, f)
    print(f"Saved {len(entry_ids):,} entry IDs to {ENTRY_IDS_PATH}")

# Normalise to lowercase — PDBe API uses lowercase IDs
entry_ids = [eid.lower() for eid in entry_ids]

print(f"\nTotal PDB entries: {len(entry_ids):,}")
print(f"First 10 IDs: {entry_ids[:10]}")

### 1.3 Fetch Entry Summaries (Sample)

Loop over the first 500 PDB IDs and retrieve a summary for each via the PDBe REST API (`/pdb/entry/summary/{pdbId}`). Each call returns title, experimental method, resolution, organism, and deposition/release dates. Results are cached in `data/pdbe_summaries_sample.json`.

In [ ]:
SAMPLE_SIZE = 500
SUMMARIES_PATH = DATA_DIR / "pdbe_summaries_sample.json"

if SUMMARIES_PATH.exists():
    print(f"Loading cached summaries from {SUMMARIES_PATH}")
    with open(SUMMARIES_PATH) as f:
        summaries_raw = json.load(f)
else:
    sample_ids = entry_ids[:SAMPLE_SIZE]
    summaries_raw = {}
    errors = []

    print(f"Fetching summaries for {SAMPLE_SIZE} entries ...")
    for i, pdb_id in enumerate(sample_ids):
        try:
            data = pdbe_get(f"pdb/entry/summary/{pdb_id}")
            summaries_raw[pdb_id] = data[pdb_id]
        except Exception as exc:
            errors.append((pdb_id, str(exc)))

        # Progress indicator every 50 entries
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{SAMPLE_SIZE} fetched  ({len(errors)} errors so far)")

        # Polite delay — avoid hammering the PDBe servers
        time.sleep(0.05)

    print(f"\nDone. {len(summaries_raw)} summaries retrieved, {len(errors)} errors.")
    if errors:
        print(f"Failed IDs: {[e[0] for e in errors[:10]]}")

    with open(SUMMARIES_PATH, "w") as f:
        json.dump(summaries_raw, f)
    print(f"Saved to {SUMMARIES_PATH}")

print(f"\nSample size in cache: {len(summaries_raw):,} entries")

### 1.4 Parse into Polars DataFrame

Extract key fields from the raw summary JSON and assemble a tidy Polars DataFrame. Each row is one PDB entry.

| Column | Source field | Notes |
|---|---|---|
| `pdb_id` | dict key | Lowercase 4-character accession |
| `title` | `title` | Free-text structure title |
| `experimental_method` | `experimental_method` | List → joined string (e.g. `"X-ray diffraction"`) |
| `resolution` | `resolution` | Float angstroms; `null` for NMR/EM entries without a value |
| `deposition_date` | `deposition_date` | String `YYYYMMDD` — parsed to `Date` in section 2 |
| `release_date` | `release_date` | String `YYYYMMDD` |
| `organism` | `organism_scientific_name` inside `related_structures` / top-level | Scientific name of the source organism |

In [ ]:
def extract_organism(entry: dict) -> str | None:
    """
    Pull the scientific name of the first source organism from an entry summary.

    PDBe stores organism information inside the `related_structures` list when
    present, but the most reliable top-level field is `organism_scientific_name`
    introduced in later API versions. We fall back gracefully.

    Parameters
    ----------
    entry : dict
        One element of the list returned by ``/pdb/entry/summary/{pdbId}``.

    Returns
    -------
    str or None
        Scientific name string, or None if not available.
    """
    # Preferred: top-level field (present in most modern responses)
    if entry.get("organism_scientific_name"):
        names = entry["organism_scientific_name"]
        if isinstance(names, list) and names:
            return names[0]
        if isinstance(names, str):
            return names

    # Fallback: look inside related_structures (older schema)
    for rs in entry.get("related_structures", []):
        name = rs.get("organism_scientific_name")
        if name:
            return name if isinstance(name, str) else name[0]

    return None


def parse_summaries(raw: dict) -> pl.DataFrame:
    """
    Convert the raw dict of PDBe summary responses into a Polars DataFrame.

    Parameters
    ----------
    raw : dict
        Mapping of {pdb_id: [entry_dict, ...]} as returned by the PDBe API
        and stored in the cache file.

    Returns
    -------
    pl.DataFrame
        One row per PDB entry with columns:
        pdb_id, title, experimental_method, resolution,
        deposition_date, release_date, organism.
    """
    records = []
    for pdb_id, entries in raw.items():
        if not entries:
            continue
        e = entries[0]  # PDBe returns a list; take the first (and usually only) element

        # experimental_method may be a list (e.g. ["X-ray diffraction"]) or a string
        methods = e.get("experimental_method", [])
        if isinstance(methods, list):
            method_str = "; ".join(methods)
        else:
            method_str = str(methods)

        records.append({
            "pdb_id": pdb_id,
            "title": e.get("title"),
            "experimental_method": method_str or None,
            "resolution": e.get("resolution"),          # float or None
            "deposition_date": e.get("deposition_date"),
            "release_date": e.get("release_date"),
            "organism": extract_organism(e),
        })

    df = pl.DataFrame(
        records,
        schema={
            "pdb_id": pl.Utf8,
            "title": pl.Utf8,
            "experimental_method": pl.Utf8,
            "resolution": pl.Float64,
            "deposition_date": pl.Utf8,
            "release_date": pl.Utf8,
            "organism": pl.Utf8,
        },
    )
    return df


df = parse_summaries(summaries_raw)

print(f"DataFrame shape: {df.shape}")
print(f"\nColumn dtypes:")
for col, dtype in zip(df.columns, df.dtypes):
    null_count = df[col].null_count()
    null_pct = null_count / len(df) * 100
    print(f"  {col:<22} {str(dtype):<12}  nulls: {null_count:>4} ({null_pct:.1f}%)")

print()
df.head(5)